In [ ]:
import requests

#авторизация на MOEX ISS через basic-аутентификацию
MOEX_LOGIN = 'orlovapolya0211@gmail.com' #логин от moex.com
MOEX_PASSWORD = 'E!3UStXK#wAwLgh'  #пароль от moex.com

session = requests.Session()
avtoriz_otvet = session.get(
    'https://passport.moex.com/authenticate',
    auth=(MOEX_LOGIN, MOEX_PASSWORD))

if avtoriz_otvet.status_code == 200:
    print('Авторизация успешна')
    print(f'Cookie получен: {"MicexPassportCert" in session.cookies}')
else:
    print(f'Ошибка авторизации: {avtoriz_otvet.status_code}')

Авторизация успешна
Cookie получен: True


In [ ]:
#cоздаём конфигурационный файл parametrs.txt
#параметры берём из статьи Tproger
#там написано - у basicConfig() три основных параметра  - level, filename, format

parametrs_content = """enabled=true
level=INFO
filename=logs/logging_infa
format=%(asctime)s - %(levelname)s - %(lineno)d - %(message)s
"""
#enabled=true/false  - включение/выключение logging без изменения кода
#level=INFO
#filename=logs/logging_infa - где хранятся наши логи
#Tproger: format=%(asctime)s - %(levelname)s - %(lineno)d - %(message)s
#убрали funcName потому что у нас нет функций
#%(asctime)s   - дата и время события
#%(levelname)s  - уровень: INFO в нашем случае
#%(message)s   - само сообщение
#%(lineno)d - номер строки


with open('parametrs.txt', 'w') as filik:
    filik.write(parametrs_content)

print('создали parametrs.txt с такими параметрами:')
with open('parametrs.txt') as filik:
    print(filik.read())

создали parametrs.txt с такими параметрами:
enabled=true
level=INFO
filename=logs/logging_infa
format=%(asctime)s - %(levelname)s - %(lineno)d - %(message)s



In [ ]:
!mkdir -p logs #создали папку, где будет файл с логами

In [ ]:
import logging

#читаем наши настройки из parametrs.txt
with open('parametrs.txt') as filik:
    lines = filik.readlines()

enabled = lines[0].split('=')[1].strip()
level_str = lines[1].split('=')[1].strip()
filename = lines[2].split('=')[1].strip()
log_format = lines[3].split('=')[1].strip()

if enabled == 'true':

    #сбрасываем старые handlers
    #потому что иначе basicConfig игнорируется в Colab
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    #filename  - из parametrs.txt
    #из Хабра нашли, что записываем сообщения в файл
    #файл будет хранить данные после завершения программы

    #format  - из parametrs.txt
    #Tproger: format с %(asctime)s %(levelname)s %(message)s

    #запись в файл
    #Tproger: 'logging.basicConfig()  - основная функция для настройки logging'
    #уровень INFO  - вывод данных о фрагментах кода, работающих так как ожидается
    logging.basicConfig(level=logging.INFO, filename=filename, format=log_format, filemode='a')
    #filemode='a'  - логи добавляются в конец файла и не стираются при перезапуске

    logging.info('Подключаем logging и собираем данные с MOEX ISS API')
    print(f'Logging работает: уровень={level_str}, файл={filename}')

else:
    print('Logging не работает')

Logging работает: уровень=INFO, файл=logs/logging_infa


In [ ]:
import requests #сначала все запросы писали через requests.get() напрямую, потом переписали через session для логгирования
import pandas as pd

Импортируем две библиотеки:
- **requests**  - для отправки HTTP-запросов к API биржи (для колаба без логгирования нужна была)
- **pandas**  - для работы с таблицами. Превращаем неудобный JSON в удобный DataFrame

## Запрос 1  - Дневные свечи (candles)

**Цель:** получить историю дневных цен Роснефти за период 2018-01-01  - 2025-12-31.

Это главные данные всего проекта. Без них невозможно посчитать нашу таргетную переменную  - изменение цены акции через 24 часа после выхода новости.

Каждая строка в ответе = один торговый день.


In [ ]:
#запрос 1: получаем дневные свечи РОСНЕФТЬ с MOEX (первые 500 строк)
response_ROSN = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/ROSN/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24})

data_ROSN = response_ROSN.json()

#извлекаем данные из json
columns_ROSN = data_ROSN['candles']['columns']
rows_ROSN = data_ROSN['candles']['data']

df_ROSN = pd.DataFrame(rows_ROSN, columns=columns_ROSN)
df_ROSN['ticker'] = 'ROSN'

print(f'Count of rows: {len(df_ROSN)}')
#считаем все строки  - мало ли их много, а выводит только 500
df_ROSN


Count of rows: 500


,open,close,high,low,value,volume,begin,end,ticker
0,292.25,297.95,297.95,292.15,6.594783e+08,2232020,2018-01-03 00:00:00,2018-01-03 23:59:59,ROSN
1,298.95,307.00,308.25,297.60,1.754455e+09,5762200,2018-01-04 00:00:00,2018-01-04 23:59:59,ROSN
2,307.00,311.95,313.65,304.65,1.312501e+09,4237350,2018-01-05 00:00:00,2018-01-05 23:59:59,ROSN
3,312.60,315.10,316.25,311.15,1.438608e+09,4577460,2018-01-09 00:00:00,2018-01-09 23:59:59,ROSN
4,316.10,317.75,319.85,312.00,1.758738e+09,5564030,2018-01-10 00:00:00,2018-01-10 23:59:59,ROSN
...,...,...,...,...,...,...,...,...,...
495,450.80,448.45,452.65,448.00,1.617101e+09,3597230,2019-12-16 00:00:00,2019-12-16 23:59:59,ROSN
496,449.95,454.05,455.00,449.20,2.360874e+09,5215360,2019-12-17 00:00:00,2019-12-17 23:59:59,ROSN
497,453.00,454.45,456.45,451.55,1.993584e+09,4389380,2019-12-18 00:00:00,2019-12-18 23:59:59,ROSN
498,454.50,451.75,456.85,449.00,2.309586e+09,5095820,2019-12-19 00:00:00,2019-12-19 23:59:59,ROSN


**Результат:**  Count of rows: 500

Таблица содержит 9 колонок:
-  open   - цена акции Роснефти в начале торгового дня (рублей за акцию)
-  close   - цена в конце торгового дня
-  high   - максимальная цена за день
-  low   - минимальная цена за день
-  value   - суммарный оборот за день в рублях (сколько рублей прошло через все сделки)
-  volume   - количество акций сменивших владельца за день
-  begin   - дата и время начала торгового дня
-  end   - дата и время конца торгового дня
-  ticker   - тикер бумаги (добавили сами, чтобы потом объединить все 7 компаний)


Также из документации (стр. 11) узнали что сервер возвращает данные порциями. Чтобы выгрузить полный объём данных, нужно последовательно делать запросы, увеличивая параметр start на значение pagesize, пока не получим пустой блок данных.



In [ ]:
logging.info('Запрос 1: проверяем есть ли данные после 500 строки (start=500)')
#проверяем есть ли данные ниже 500 строки
response_ROSN = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/ROSN/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 500
      })

data_ROSN = response_ROSN.json()
print(f"Count of rows after 500 rows: {len(data_ROSN['candles']['data'])}")
#строк много  - выводим оставшиеся


Count of rows after 500 rows: 500


**Результат:**  Count of rows after 500 rows: 500

HOOORRAY!!))

Ну вот видите?)

Данные не закончились на 500-й строке. Получили ещё 500 строк  - это оставшиеся торговые дни, которые не влезли в первый запрос.

Теперь будем проверять, пока нам не выведет 0 значений.

In [ ]:
columns_ROSNnew = data_ROSN['candles']['columns']
rows_ROSNnew = data_ROSN['candles']['data']

df_ROSNnew = pd.DataFrame(rows_ROSNnew, columns=columns_ROSNnew)
df_ROSNnew['ticker'] = 'ROSN'

print(f'Count of rows after 500 rows: {len(df_ROSNnew)}')

#проверяем есть ли данные ниже 1000 строки
logging.info('Запрос 1: проверяем есть ли данные после 1000 строки (start=1000)')
response_ROSN2 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/ROSN/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 1000
      })

data_ROSN2 = response_ROSN2.json()
columns_ROSNnew2 = data_ROSN2['candles']['columns']
rows_ROSNnew2 = data_ROSN2['candles']['data']

df_ROSNnew2 = pd.DataFrame(rows_ROSNnew2, columns=columns_ROSNnew2)
df_ROSNnew2['ticker'] = 'ROSN'

print(f'Count of rows after 1000 rows: {len(df_ROSNnew2)}')

#проверяем есть ли данные ниже 1500 строки
logging.info('Запрос 1: проверяем есть ли данные после 1500 строки (start=1500)')
response_ROSN3 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/ROSN/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 1500
      })

data_ROSN3 = response_ROSN3.json()
columns_ROSNnew3 = data_ROSN3['candles']['columns']
rows_ROSNnew3 = data_ROSN3['candles']['data']

df_ROSNnew3 = pd.DataFrame(rows_ROSNnew3, columns=columns_ROSNnew3)
df_ROSNnew3['ticker'] = 'ROSN'

print(f'Count of rows after 1500 rows: {len(df_ROSNnew3)}')



Count of rows after 500 rows: 500
Count of rows after 1000 rows: 500
Count of rows after 1500 rows: 500


In [ ]:
#проверяем есть ли данные ниже 2000 строки

logging.info('Запрос 1: проверяем есть ли данные после 2000 строки (start=2000)')
response_ROSN4 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/ROSN/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 2000
      })

data_ROSN4 = response_ROSN4.json()
columns_ROSNnew4 = data_ROSN4['candles']['columns']
rows_ROSNnew4 = data_ROSN4['candles']['data']

df_ROSNnew4 = pd.DataFrame(rows_ROSNnew4, columns=columns_ROSNnew4)
df_ROSNnew4['ticker'] = 'ROSN'

print(f'Count of rows after 2000 rows: {len(df_ROSNnew4)}')

Count of rows after 2000 rows: 73


In [ ]:
#проверяем есть ли данные ниже 2073 строки
logging.info('Запрос 1: проверяем есть ли данные после 2073 строки (start=2073)')
response_ROSN5 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/ROSN/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 2073
      })

data_ROSN5 = response_ROSN5.json()
columns_ROSNnew5 = data_ROSN5['candles']['columns']
rows_ROSNnew5 = data_ROSN5['candles']['data']

df_ROSNnew5 = pd.DataFrame(rows_ROSNnew5, columns=columns_ROSNnew5)
df_ROSNnew5['ticker'] = 'ROSN'

print(f'Count of rows after 2073 rows: {len(df_ROSNnew5)}')

Count of rows after 2073 rows: 0


In [ ]:
#больше данных нет, поэтому обьединяем датафреймы в один
df_ROSNcandles = pd.concat([df_ROSN, df_ROSNnew, df_ROSNnew2, df_ROSNnew3, df_ROSNnew4], ignore_index=True)
print(f'Total rows: {len(df_ROSNcandles)}')

logging.info(f'Запрос 1: итого после склейки всех частей пяти датафреймов  - {len(df_ROSNcandles)} строк')
df_ROSNcandles

Total rows: 2073


,open,close,high,low,value,volume,begin,end,ticker
0,292.25,297.95,297.95,292.15,6.594783e+08,2232020,2018-01-03 00:00:00,2018-01-03 23:59:59,ROSN
1,298.95,307.00,308.25,297.60,1.754455e+09,5762200,2018-01-04 00:00:00,2018-01-04 23:59:59,ROSN
2,307.00,311.95,313.65,304.65,1.312501e+09,4237350,2018-01-05 00:00:00,2018-01-05 23:59:59,ROSN
3,312.60,315.10,316.25,311.15,1.438608e+09,4577460,2018-01-09 00:00:00,2018-01-09 23:59:59,ROSN
4,316.10,317.75,319.85,312.00,1.758738e+09,5564030,2018-01-10 00:00:00,2018-01-10 23:59:59,ROSN
...,...,...,...,...,...,...,...,...,...
2068,402.85,408.50,408.75,402.85,1.103843e+09,2717657,2025-12-26 00:00:00,2025-12-26 23:59:58,ROSN
2069,408.30,409.60,410.10,408.00,2.222036e+08,542953,2025-12-27 00:00:00,2025-12-27 23:59:57,ROSN
2070,410.00,411.00,411.25,407.60,1.678375e+08,409619,2025-12-28 00:00:00,2025-12-28 23:59:57,ROSN
2071,410.45,408.65,418.50,406.50,2.935446e+09,7100581,2025-12-29 00:00:00,2025-12-29 23:59:59,ROSN


Hooray!

Пустой ответ подтверждает, что мы выгрузили абсолютно все данные. После 2073 строки данных нет. Значит итоговый датафрейм  df_ROSNcandles  содержит полную историю дневных котировок Роснефти за весь нужный период без пропусков.

**Результат:**
   
Total rows: 2073
   

Склеили пять кусков через  pd.concat  с параметром  ignore_index=True   - пересчитаем индексы строк заново от 0, иначе в итоговой таблице было бы пять наборов индексов, что вызвало бы у нас путаницу и отсутствие hoorray(.

Итого получили **все торговые дни за период**  - это все торговые дни Роснефти за период с 01.01.2018 по 31.12.2025.



## Запрос 2  - Информация о бумаге (security info)

**Цель:** получить полную информацию по акциям Роснефти: официальное название компании, международный код бумаги (ISIN), номинальная стоимость акции и уровень листинга.

Эти данные нужны нам для двух вещей:
1. Красиво описать датасет в README  - чтобы было понятно с какими именно бумагами работаем
2. Добавить уровень листинга как признак в EDA  - бумаги первого уровня листинга торгуются активнее и имеют более строгие требования к раскрытию информации

In [ ]:
#запрос 2: получаем статические характеристики бумаги РОСНЕФТЬ
#нам возвращается: полное название, ISIN, номинал, уровень листинга, тип бумаги
response_ROSN_infa = session.get(
    'https://iss.moex.com/iss/securities/ROSN.json')

data_ROSN_infa = response_ROSN_infa.json()

#раздел 'description' содержит характеристики в виде строк name/value
columns_ROSN_infa = data_ROSN_infa['description']['columns']
rows_ROSN_infa = data_ROSN_infa['description']['data']

df_ROSN_infa = pd.DataFrame(rows_ROSN_infa, columns=columns_ROSN_infa)
df_ROSN_infa['ticker'] = 'ROSN'

print(f'Count of rows: {len(df_ROSN_infa)}')
df_ROSN_infa

Count of rows: 27


,name,title,value,type,sort_order,is_hidden,precision,ticker
0,SECID,Код ценной бумаги,ROSN,string,1,0,NaN,ROSN
1,ISSUENAME,Наименование ценной бумаги,Акции обыкновенные,string,2,0,NaN,ROSN
2,NAME,Полное наименование,ПАО НК Роснефть,string,3,0,NaN,ROSN
3,SHORTNAME,Краткое наименование,Роснефть,string,4,0,NaN,ROSN
4,ISIN,ISIN код,RU000A0J2Q06,string,5,0,NaN,ROSN
5,REGNUMBER,Номер государственной регистрации,1-02-00122-A,string,6,0,NaN,ROSN
6,ISSUESIZE,Объем выпуска,10598177817,number,7,0,0.0,ROSN
7,FACEVALUE,Номинальная стоимость,0.01,number,8,0,2.0,ROSN
8,FACEUNIT,Валюта номинала,SUR,string,9,0,NaN,ROSN
9,ISSUEDATE,Дата начала торгов,2006-07-19,date,10,0,NaN,ROSN


**Результат:**  Count of rows: 27

Получили 27 строк.

Каждая строка это одна характеристика бумаги.

Каждая характеристика на отдельной строке.

Это неудобно для анализа, поэтому в следующей ячейке мы преобразуем это формат, где одна строка на компанию, каждая характеристика отдельная колонка.

In [ ]:
#преобразуем в удобный формат: одна строка  - один тикер
#разворачиваем колонку 'name' в отдельные колонки через pivot
df_ROSN_infa_best = df_ROSN_infa.set_index('name')['value'].to_frame().T
df_ROSN_infa_best['ticker'] = 'ROSN'
df_ROSN_infa_best = df_ROSN_infa_best.reset_index()

print(f'Полное название: {df_ROSN_infa_best["NAME"].values[0]}')
print(f'ISIN: {df_ROSN_infa_best["ISIN"].values[0]}')
print(f'Уровень листинга: {df_ROSN_infa_best["LISTLEVEL"].values[0]}')

logging.info(f'Запрос 2: ISIN={df_ROSN_infa_best["ISIN"].values[0]}, листинг={df_ROSN_infa_best["LISTLEVEL"].values[0]}')

df_ROSN_infa_best

Полное название: ПАО НК Роснефть
ISIN: RU000A0J2Q06
Уровень листинга: 1


name,index,SECID,ISSUENAME,NAME,SHORTNAME,ISIN,REGNUMBER,ISSUESIZE,FACEVALUE,FACEUNIT,...,MORNINGSESSION,EVENINGSESSION,WEEKENDSESSION,REGISTRY_DATE,TYPENAME,GROUP,TYPE,GROUPNAME,EMITTER_ID,ticker
0,value,ROSN,Акции обыкновенные,ПАО НК Роснефть,Роснефть,RU000A0J2Q06,1-02-00122-A,10598177817,0.01,SUR,...,1,1,1,2005-09-29,Акция обыкновенная,stock_shares,common_share,Акции,651,ROSN


**Результат:**
   
Полное название: ПАО НК Роснефть
ISIN: RU000A0J2Q06
Уровень листинга: 1
   

Теперь это одна строка с колонками NAME, ISIN, LISTLEVEL и тд.

Что значат выводы:
- **NAME = «ПАО НК Роснефть»**  - официальное юридическое название компании.
- **ISIN = RU000A0J2Q06**  - международный идентификационный номер ценной бумаги.
- **LISTLEVEL = 1**  - первый уровень листинга на Мосбирже. Это значит, что Роснефть прошёл самые строгие требования биржи по прозрачности и раскрытию информации.

**Итог запроса 2:** получили официальную информацию по бумаге.

Hooray!))

## Запрос 3  - История дивидендов (dividends)

**Цель:** получить все даты дивидендных выплат Роснефти за период проекта.

Зачем нам это нужно: когда компания выплачивает дивиденды, цена акции резко падает примерно на размер дивиденда.

Если мы не пометим эти дни, то можем ошибочно решить в EDA, что резкое падение цены вызвано негативной новостью  - хотя на самом деле это просто плановая выплата. Это могло бы нам помешать.

In [ ]:
logging.info('Запрос 3: загружаем историю дивидендов ROSN')
#запрос 3: получаем историю дивидендных выплат РОСНЕФТЬ
#дивиденды влияют на котировки примерно на размер дивиденда
ROSN_diva = session.get(
    'https://iss.moex.com/iss/securities/ROSN/dividends.json')

ROSN_diva_infa = ROSN_diva.json()

#извлекаем данные из json
columns_ROSN_diva = ROSN_diva_infa['dividends']['columns']
rows_ROSN_diva = ROSN_diva_infa['dividends']['data']

df_ROSN_diva = pd.DataFrame(rows_ROSN_diva, columns=columns_ROSN_diva)
df_ROSN_diva['ticker'] = 'ROSN'

print(f'Count of rows: {len(df_ROSN_diva)}')

logging.info(f'Запрос 3: получено {len(df_ROSN_diva)} дивидендных выплат')
df_ROSN_diva

Count of rows: 19


,secid,isin,registryclosedate,value,currencyid,ticker
0,ROSN,RU000A0J2Q06,2014-07-08,12.85,RUB,ROSN
1,ROSN,RU000A0J2Q06,2015-06-29,8.21,RUB,ROSN
2,ROSN,RU000A0J2Q06,2016-06-27,11.75,RUB,ROSN
3,ROSN,RU000A0J2Q06,2017-07-03,5.98,RUB,ROSN
4,ROSN,RU000A0J2Q06,2017-10-10,3.83,RUB,ROSN
5,ROSN,RU000A0J2Q06,2018-07-02,6.65,RUB,ROSN
6,ROSN,RU000A0J2Q06,2018-10-09,14.58,RUB,ROSN
7,ROSN,RU000A0J2Q06,2019-06-17,11.33,RUB,ROSN
8,ROSN,RU000A0J2Q06,2019-10-11,15.34,RUB,ROSN
9,ROSN,RU000A0J2Q06,2020-06-15,18.07,RUB,ROSN


**Результат:**  Count of rows: 19

API вернул все выплаты дивидентов за все годы существования на бирже. Таблица содержит колонки:

- **registryclosedate**  - дата закрытия реестра.
- **value**  - размер дивиденда на одну акцию в рублях.

Нам нужны только выплаты за наш период 2018–2025, поэтому сейчас будет фильтровать.

In [ ]:
#фильтруем дивиденды по диапазону дат проекта 2018-01-01 - 2025-12-31
df_ROSN_diva['registryclosedate'] = pd.to_datetime(df_ROSN_diva['registryclosedate'])
df_ROSN_diva_srok = df_ROSN_diva[
    (df_ROSN_diva['registryclosedate'] >= '2018-01-01') &
    (df_ROSN_diva['registryclosedate'] <= '2025-12-31')].reset_index()

print(f'Все дивиденты за 2018-01-01 - 2025-12-31: {len(df_ROSN_diva_srok)}')

logging.info(f'Запрос 2: ISIN={df_ROSN_infa_best["ISIN"].values[0]}, листинг={df_ROSN_infa_best["LISTLEVEL"].values[0]}')
df_ROSN_diva_srok

Все дивиденты за 2018-01-01 - 2025-12-31: 14


,index,secid,isin,registryclosedate,value,currencyid,ticker
0,5,ROSN,RU000A0J2Q06,2018-07-02,6.65,RUB,ROSN
1,6,ROSN,RU000A0J2Q06,2018-10-09,14.58,RUB,ROSN
2,7,ROSN,RU000A0J2Q06,2019-06-17,11.33,RUB,ROSN
3,8,ROSN,RU000A0J2Q06,2019-10-11,15.34,RUB,ROSN
4,9,ROSN,RU000A0J2Q06,2020-06-15,18.07,RUB,ROSN
5,10,ROSN,RU000A0J2Q06,2021-06-15,6.94,RUB,ROSN
6,11,ROSN,RU000A0J2Q06,2021-10-11,18.03,RUB,ROSN
7,12,ROSN,RU000A0J2Q06,2022-07-11,23.63,RUB,ROSN
8,13,ROSN,RU000A0J2Q06,2023-01-12,20.39,RUB,ROSN
9,14,ROSN,RU000A0J2Q06,2023-07-11,17.97,RUB,ROSN


**Результат:**  Все дивиденты за 2018-01-01 - 2025-12-31: 14


Почему конвертировали дату через  pd.to_datetime : без этого Python сравнивал бы строки а не даты. Строковое сравнение работает по алфавиту и даёт неверные результаты для дат в формате YYYY-MM-DD.

**Итог запроса 3:** знаем точные даты дивидендных выплат. В EDA пометим эти дни флагом  is_dividend_day=1 .

## Запрос 4  - Текущие рыночные данные (market data)

Запрос 4 получает текущие торговые данные по Роснефти  - последнюю цену, объём торгов за день, количество сделок и капитализацию компании. Всё это показывает насколько активно торгуется акция прямо сейчас.


Эти данные характеризуют **ликвидность** бумаги  - насколько активно она торгуется. В EDA мы будем использовать их чтобы показать: более ликвидные бумаги быстрее и точнее реагируют на новости, потому что больше участников рынка мгновенно перекладываются после публикации статьи.

In [ ]:
logging.info('Запрос 4: загружаем текущие рыночные данные ROSN')
#запрос 4: получаем текущие торговые данные РОСНЕФТЬ
response_ROSN_markdata = session.get(
    'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/ROSN.json')

data_ROSN_markdata = response_ROSN_markdata.json()


#раздел 'marketdata' содержит торговые данные текущей/последней сессии
columns_ROSN_markdata = data_ROSN_markdata['marketdata']['columns']
rows_ROSN_markdata = data_ROSN_markdata['marketdata']['data']

df_ROSN_markdata = pd.DataFrame(rows_ROSN_markdata, columns=columns_ROSN_markdata)
df_ROSN_markdata['ticker'] = 'ROSN'

print(f'Count of rows: {len(df_ROSN_markdata)}')

logging.info(f'Запрос 4: получено {len(df_ROSN_markdata)} строк market data')
df_ROSN_markdata

Count of rows: 1


,SECID,BOARDID,BID,BIDDEPTH,OFFER,OFFERDEPTH,SPREAD,BIDDEPTHT,OFFERDEPTHT,OPEN,...,SYSTIME,CLOSINGAUCTIONPRICE,CLOSINGAUCTIONVOLUME,ISSUECAPITALIZATION,ISSUECAPITALIZATION_UPDATETIME,ETFSETTLECURRENCY,VALTODAY_RUR,TRADINGSESSION,TRENDISSUECAPITALIZATION,ticker
0,ROSN,TQBR,510,None,510.05,None,0.05,1474323,2437061,513,...,2026-03-19 22:48:13,513.5,45889,5.414079e+12,22:46:57,None,20129736032,2,-1.430754e+10,ROSN


**Результат:**  Count of rows: 1

Одна строка  - потому что это текущие торговые данные. Но колонок очень много. API возвращает абсолютно всё что знает о торговле этой бумагой прямо сейчас.

Большинство колонок нам не нужны  - это технические поля биржи вроде кодов режимов торгов, флагов состояния и прочего. В следующей ячейке оставим только то что важно для нашего анализа.

In [ ]:
#оставляем только нужные колонки из market data
#используем фильтрацию через список  - так код не упадёт если какой-то колонки нет в ответе
markdata_cols = ['ticker', 'LAST', 'LASTTOPREVPRICE', 'BID', 'OFFER',
            'OPEN', 'HIGH', 'LOW', 'WAPRICE', 'NUMTRADES',
            'VOLTODAY', 'VALTODAY', 'ISSUECAPITALIZATION', 'UPDATETIME']
markdata_cols = [col for col in markdata_cols if col in df_ROSN_markdata.columns]

df_ROSN_markdata_best = df_ROSN_markdata[markdata_cols].copy()

#выводим ключевые показатели
print(f'Последняя цена ROSN: {df_ROSN_markdata_best["LAST"].values[0]}')
print(f'Изменение к пред. закрытию: {df_ROSN_markdata_best["LASTTOPREVPRICE"].values[0]}')
print(f'Капитализация: {df_ROSN_markdata_best["ISSUECAPITALIZATION"].values[0]}')

logging.info(f'Запрос 4: последняя цена ROSN={df_ROSN_markdata_best["LAST"].values[0]}')

df_ROSN_markdata_best

Последняя цена ROSN: 510.05
Изменение к пред. закрытию: -0.42
Капитализация: 5414079137814.45


,ticker,LAST,LASTTOPREVPRICE,BID,OFFER,OPEN,HIGH,LOW,WAPRICE,NUMTRADES,VOLTODAY,VALTODAY,ISSUECAPITALIZATION,UPDATETIME
0,ROSN,510.05,-0.42,510,510.05,513,518.7,507.8,513.8,194798,39174217,20129736032,5.414079e+12,22:33:13


**Результат:** (тут данные могут изменяться при каждом запуске)
   
Последняя цена ROSN: 509.3
Изменение к пред. закрытию: -0.57
Капитализация: 5410899684469.35
   

Что означают колонки которые мы оставили:
- **LAST**  - последняя цена сделки (рублей за акцию)
- **LASTTOPREVPRICE**  - изменение в % к цене закрытия предыдущего дня
- **BID / OFFER**  - лучшая цена покупки и продажи прямо сейчас. Разница между ними называется спредом  - чем он уже, тем ликвиднее бумага
- **WAPRICE**  - средневзвешенная цена за весь день (учитывает объём каждой сделки)
- **NUMTRADES**  - количество сделок за торговый день. У Роснефти это десятки тысяч  - одна из самых торгуемых бумаг на бирже
- **VOLTODAY / VALTODAY**  - объём в акциях и оборот в рублях за сегодня
- **ISSUECAPITALIZATION**  - рыночная капитализация в рублях.

**Итог запроса 4:** получили характеристику ликвидности Роснефти на текущий момент.

## Запрос 5  - Индексная принадлежность (indices)

**Цель:** узнать в какие биржевые индексы входит Роснефть.

Индексные бумаги  - особая категория. Когда акция входит в IMOEX (Индекс МосБиржи), за ней автоматически следят индексные фонды (ETF) и управляющие компании. Это означает что при выходе важной новости реагируют не только частные инвесторы, но и алгоритмы фондов.

В EDA мы сравним: одинаково ли сильно реагируют на новости индексные бумаги и менее крупные компании из нашего списка.

In [ ]:
logging.info('Запрос 5: загружаем индексы ROSN')
#запрос 5: получаем список биржевых индексов в которые входит РОСНЕФТЬ
response_ROSN_idishka = session.get(
    'https://iss.moex.com/iss/securities/ROSN/indices.json')

data_ROSN_idishka = response_ROSN_idishka.json()

#извлекаем данные из json
columns_ROSN_idishka = data_ROSN_idishka['indices']['columns']
rows_ROSN_idishka = data_ROSN_idishka['indices']['data']

df_ROSN_idishka = pd.DataFrame(rows_ROSN_idishka, columns=columns_ROSN_idishka)
df_ROSN_idishka['ticker'] = 'ROSN'

print(f'Count of rows: {len(df_ROSN_idishka)}')

logging.info(f'Запрос 5: ROSN входит в {len(df_ROSN_idishka)} индексов')

df_ROSN_idishka

Count of rows: 20


,SECID,SHORTNAME,FROM,TILL,CURRENTVALUE,LASTCHANGEPRC,LASTCHANGE,ticker
0,EPSI,Субиндекс акций,2014-09-16,2026-03-18,1665.38,0.00,-0.02,ROSN
1,HCAP,HCAP,2024-06-21,2024-09-26,907.67,-0.07,-0.66,ROSN
2,ICLIMATE,Индекс МосБиржи Климатический,2025-07-01,2026-03-18,1030.83,0.23,2.39,ROSN
3,IMOEXCNY,Индекс МосБиржи в юанях,2024-09-24,2026-03-18,1060.38,-2.49,-27.11,ROSN
4,IMOEXW,IMOEXW,2023-01-16,2026-03-18,2881.58,-0.15,-4.34,ROSN
5,MESG,Индекс МосБиржи-RAEX ESG,2024-01-24,2026-03-18,984.75,-0.27,-2.69,ROSN
6,MOEXBC,Индекс голубых фишек,2010-01-14,2026-03-19,19057.56,-0.03,-5.43,ROSN
7,MRRT,Индекс MRRT,2020-09-21,2026-03-18,2335.80,0.05,1.15,ROSN
8,MRSV,Индекс MRSV,2020-09-21,2026-03-18,2173.86,-0.22,-4.80,ROSN
9,MRSVR,Индекс МосБиржи-РСПП MRSV RU Co,2021-03-25,2026-03-18,2237.45,-0.22,-4.93,ROSN


**Результат:**  Count of rows: 19


Важные колонки:
- **SECID**  - код индекса
- **FROM / TILL**  - период с которого по который бумага входит в индекс
- **CURRENTVALUE**  - текущее значение самого индекса

In [ ]:
#проверяем входит ли ROSN в IMOEX (главный российский фондовый индекс)
if len(df_ROSN_idishka) > 0:
    imoex_check = df_ROSN_idishka[df_ROSN_idishka['SECID'] == 'IMOEX']
    print(f'ROSN входит в IMOEX: {len(imoex_check) > 0}')
else:
    print('ROSN не входит ни в один индекс MOEX')
logging.info(f'Запрос 5 завершили  - все данные по ROSN собраны')


ROSN входит в IMOEX: True


**Результат:**
   
ROSN входит в IMOEX: True
   

Роснефть входит в IMOEX  - это подтверждает что он является влиятельной компанией российского рынка.

**Итог запроса 5:** Роснефть входит в IMOEX и другие индексы Мосбиржи. В EDA это будет важным признаком при анализе силы реакции на новости.



